In [112]:
from scipy.optimize import linprog

# 1. Цільова функція (помножена на -1 для мінімізації замість максимізації)
c = [-60, -45]

# 2. Обмеження-нерівності типу (<=) [A_ub * x <= b_ub]
A_ub = [
    [3, 5],    # сировина
    [4, 2],    # робочий час
    [20, 10]   # електроенергія
]
b_ub = [300, 220, 1195]

# 3. Межі змінних (x1 >= 0, x2 >= 0)
bounds = [(0, None), (0, None)]

# 4. Розв'язання задачі методом highs
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print(f"Статус виконання (status == 0 означає успіх): {res.status}")
print(f"Повідомлення: {res.message}")
print(f"Оптимальні кількості (x1, x2): {res.x}")
print(f"Мінімізована цільова функція (-res.fun це максимальний прибуток): {-res.fun}")

Статус виконання (status == 0 означає успіх): 0
Повідомлення: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Оптимальні кількості (x1, x2): [35.71428571 38.57142857]
Мінімізована цільова функція (-res.fun це максимальний прибуток): 3878.5714285714284


In [113]:
opt_x1 = res.x[0]
opt_x2 = res.x[1]
max_profit = -res.fun

print(f"Оптимальний випуск виробу А: {opt_x1:.2f} од./тиждень")
print(f"Оптимальний випуск виробу Б: {opt_x2:.2f} од./тиждень")
print(f"Максимальний тижневий прибуток: {max_profit:.2f} грн")

Оптимальний випуск виробу А: 35.71 од./тиждень
Оптимальний випуск виробу Б: 38.57 од./тиждень
Максимальний тижневий прибуток: 3878.57 грн


In [114]:
# Розрахунок використаних ресурсів та залишків (slack)
resource_limits = np.array(b_ub)
resource_used = np.dot(A_ub, res.x)
slacks = resource_limits - resource_used

resources_names = ["Сировина (кг)", "Робочий час (год)", "Електроенергія (кВт·год)"]

for name, limit, used, slack in zip(resources_names, resource_limits, resource_used, slacks):
    status = "АКТИВНЕ (вузьке місце)" if abs(slack) < 1e-5 else f"НЕАКТИВНЕ (запас = {slack:.2f})"
    print(f"{name}: ліміт = {limit}, використано = {used:.2f} -> {status}")

Сировина (кг): ліміт = 300, використано = 300.00 -> АКТИВНЕ (вузьке місце)
Робочий час (год): ліміт = 220, використано = 220.00 -> АКТИВНЕ (вузьке місце)
Електроенергія (кВт·год): ліміт = 1195, використано = 1100.00 -> НЕАКТИВНЕ (запас = 95.00)


In [115]:
# 1. Збільшення ліміту активного ресурсу (наприклад, робочого часу на 15%: 220 -> 253)
A_ub_active = [
    [3, 5],
    [4, 2],
    [20, 10]
]
b_ub_active = [300, 220 * 1.15, 1195]

res_active = linprog(c, A_ub=A_ub_active, b_ub=b_ub_active, bounds=bounds, method="highs")
print(f"Збільшення активного ресурсу на 15% -> Новий прибуток: {-res_active.fun:.2f} грн (зріс на {-res_active.fun - max_profit:.2f} грн)")

# 2. Збільшення ліміту неактивного ресурсу (наприклад, сировини на 50%: 300 -> 450)
A_ub_inactive = [
    [3, 5],
    [4, 2],
    [20, 10]
]
b_ub_inactive = [300 * 1.5, 220, 1195]

res_inactive = linprog(c, A_ub=A_ub_inactive, b_ub=b_ub_inactive, bounds=bounds, method="highs")
print(f"Збільшення неактивного ресурсу на 50% -> Новий прибуток: {-res_inactive.fun:.2f} грн (зріс на {-res_inactive.fun - max_profit:.2f} грн)")

Збільшення активного ресурсу на 15% -> Новий прибуток: 4102.50 грн (зріс на 223.93 грн)
Збільшення неактивного ресурсу на 50% -> Новий прибуток: 4521.43 грн (зріс на 642.86 грн)


### Відповіді на контрольні запитання:

1. **Чому `linprog` завжди мінімізує і як розв'язувати задачу максимізації:**
   - Функція оптимізації за замовчуванням виконує пошук мінімуму. Щоб розв'язати задачу максимізації прибутку, коефіцієнти цільової функції множаться на $-1$; після знаходження розв'язку значення функції інвертується назад.

2. **Чому обмеження «не менше» ($\ge$) множаться на $-1$:**
   - Алгоритм приймає обмеження-нерівності виключно у вигляді «$\le$» (`A_ub @ x <= b_ub`). Множення нерівності типу «$\ge$» на $-1$ автоматично змінює знак у потрібний бік.

3. **Різниця між активним і неактивним обмеженням:**
   - Активне обмеження вичерпано повністю (залишок дорівнює нулю) і воно обмежує подальше зростання цільової функції (вузьке місце). Неактивне обмеження має вільний запас і не впливає на поточне оптимальне рішення.

4. **Узгодженість результатів у What-if аналізі:**
   - Збільшення ліміту активного ресурсу завжди приносить додатковий прибуток, тоді як розширення неактивного ресурсу не дає жодного ефекту, що повністю підтверджує правильність знаходження вузьких місць виробництва.